# Phase 10 — Calibration, Model Card, and Artifact Manifest

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Define score semantics, calibration evidence, model card contents, and artifact manifest requirements.

This notebook defines conservative score semantics, calibration diagnostics, model card fields, artifact manifest fields, and deployment readiness notes. It writes `reports/phase_10_calibration_model_card.json` as the machine-readable review artifact.


## Calibration boundary

Score ranges are public contract ranges, not proof of production meaning. Phase 10 records draft semantics from existing reports and explicitly blocks production claims until calibration evidence exists.

The model/core owns calibrated score values and grounded evidence. The backend/API wrapper owns product copy, persistence, job hydration, authorization, and final response mapping.


## Shared setup

### Purpose
Load generated API schema evidence, prior phase reports, and current artifact inventory for calibration, model card, and manifest planning.

### Required input
Repository root with `GAP_MODEL_TRAINING.md`, `references/docs/generated/openapi.json`, `reports/phase_00_reproducibility_snapshot.json`, `reports/phase_02_label_schema_baselines.json`, `reports/phase_05_baseline_evaluation.json`, `reports/phase_06_jobfit_training_experiments.json`, `reports/phase_07_ats_friendliness_scoring.json`, and `reports/phase_09_candidate_reranking.json`.

### Action
Read score constraints, inherited score-band policy, model output contracts, ATS benchmark policy, candidate reranking contract, and artifact inventory.

### Expected output
Reusable constants for score fields, draft band count, readiness decision, inference artifacts, and public match-level values.

### Verification
Fail fast if required inputs are missing. Confirm public score fields remain bounded and prior readiness remains no-go for production calibration.


In [7]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "GAP_MODEL_TRAINING.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"
REPORT_PATH = REPORTS / "phase_10_calibration_model_card.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 10 input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


openapi = read_json(OPENAPI_PATH)
phase0 = read_json(REPORTS / "phase_00_reproducibility_snapshot.json")
phase2 = read_json(REPORTS / "phase_02_label_schema_baselines.json")
phase5 = read_json(REPORTS / "phase_05_baseline_evaluation.json")
phase6 = read_json(REPORTS / "phase_06_jobfit_training_experiments.json")
phase7 = read_json(REPORTS / "phase_07_ats_friendliness_scoring.json")
phase9 = read_json(REPORTS / "phase_09_candidate_reranking.json")

schemas = openapi["components"]["schemas"]
score_constraints = {
    "jobFitAlignment.score": {"minimum": 0, "maximum": 100, "type": "integer"},
    "atsFriendliness.score": {"minimum": 0, "maximum": 100, "type": "integer"},
    "recommendations[].matchScore": schemas["JobRecommendationItem"]["properties"]["matchScore"],
    "jobFit.fitScore": schemas["JobFitAnalysis"]["properties"]["fitScore"],
    "recommendations[].matchLevel": schemas["JobRecommendationItem"]["properties"]["matchLevel"],
}
inference_artifacts = [a for a in phase0["artifact_inventory"] if a.get("required_for_inference")]

setup_summary = {
    "score_fields": list(score_constraints.keys()),
    "phase2_band_count": len(phase2["score_band_policy"]),
    "phase5_readiness_decision": phase5["training_readiness_gate"]["decision"],
    "legacy_inference_artifact_count": len(inference_artifacts),
    "phase9_match_levels": score_constraints["recommendations[].matchLevel"]["enum"],
}
setup_summary


{'score_fields': ['jobFitAlignment.score',
  'atsFriendliness.score',
  'recommendations[].matchScore',
  'jobFit.fitScore',
  'recommendations[].matchLevel'],
 'phase2_band_count': 3,
 'phase5_readiness_decision': 'NO_GO_FIX_PAIR_GENERATION_AND_LABELS_FIRST',
 'legacy_inference_artifact_count': 3,
 'phase9_match_levels': ['strong', 'good', 'stretch']}

## Step 10.1 — Score semantics

### Purpose
Define what score ranges mean for `jobFitAlignment`, `atsFriendliness`, and recommendation `matchScore`.

### Required input
Phase 2 score-band policy, OpenAPI score constraints, Phase 5 readiness blockers, Phase 6 job-fit output contract, Phase 7 ATS output contract, and Phase 9 reranking output contract.

### Action
Create draft score semantics from documented report evidence only. Mark every range as non-production-calibrated until bucket coverage and error metrics exist.

### Expected output
A score-semantics table for the three model-owned public score families with evidence sources and production gating rules.

### Verification
Confirm no score meaning is treated as production-ready without source reports and calibration status.


## Step 10.2 — Calibration diagnostics

### Purpose
Plan calibration plots, bucket accuracy, bucket error, and confidence notes for each output score.

### Required input
Predicted scores, trusted labels or reviewer scores, split names, score slices, and output-specific constraints.

### Action
Define 0–20, 21–40, 41–60, 61–80, and 81–100 buckets plus required tables, plots, and calibration metrics.

### Expected output
A reproducible calibration diagnostic plan covering `jobFitAlignment.score`, `atsFriendliness.score`, and `recommendations[].matchScore`.

### Verification
Confirm diagnostics include bucket count, bucket accuracy, bucket error, confidence notes, ECE, MCE, and per-output plots.


## Step 10.3 — Model card fields

### Purpose
Document model name, version, date, dataset hash, label version, split seed, feature config, metrics, limitations, and intended use.

### Required input
Exported model candidate metadata, dataset manifest, label schema, split config, feature config, metric reports, calibration report, and deployment boundary.

### Action
Define a complete model card template and output-specific metric requirements.

### Expected output
A model-card v2 template ready for future export runs.

### Verification
Confirm all required fields exist and metrics cover job-fit, ATS, recommendation, and calibration evidence.


## Step 10.4 — Artifact manifest fields

### Purpose
Document model paths, embedding/cache paths, schema versions, file hashes, and reproducibility references.

### Required input
Phase 0 artifact inventory, future export paths, model card path, feature config version, label version, split seed, dataset hash, and code version.

### Action
Define artifact manifest fields and classify current legacy inference artifacts separately from training-only or audit artifacts.

### Expected output
An artifact-manifest v1 template with hash and reproducibility policy.

### Verification
Confirm manifest fields include path, format, role, required_for_inference, schema_version, sha256, and reproducibility references.


## Step 10.5 — Deployment readiness notes

### Purpose
Describe which artifacts are required for inference and which artifacts are training-only.

### Required input
Artifact manifest template, current artifact inventory, model-card template, backend/model boundary docs, and readiness blockers from prior phases.

### Action
Separate inference-required artifact classes from training-only artifact classes and record current deployment decision.

### Expected output
Deployment readiness notes that block release until calibrated scores, model card, manifest hashes, schema versions, and inference packaging are available.

### Verification
Confirm deployment status remains not ready while calibration, labels, candidate sets, and export manifest are incomplete.


## Report generation

### Purpose
Write the complete calibration, model card, and artifact manifest design artifact.

### Required input
Shared setup constants and prior phase reports.

### Action
Build `reports/phase_10_calibration_model_card.json` with score semantics, calibration diagnostics, model card template, artifact manifest template, deployment readiness notes, blockers, and acceptance status.

### Expected output
`reports/phase_10_calibration_model_card.json`.

### Verification
The report must satisfy all acceptance criteria and include required templates, calibration metrics, and deployment blockers.


In [8]:
band_policy = phase2["score_band_policy"]

score_semantics = [
    {
        "score_name": "jobFitAlignment.score",
        "range": "0-100 integer",
        "draft_bands_from_phase2": [
            {
                "band": band["band"],
                "api_min": band["api_min"],
                "api_max": band["api_max"],
                "meaning": band["product_meaning_jobfit"],
                "evidence_required_before_production": band["training_requirement"],
            }
            for band in band_policy
        ],
        "calibration_status": "draft_semantics_only_not_production_calibrated",
        "evidence_sources": ["reports/phase_02_label_schema_baselines.json", "reports/phase_05_baseline_evaluation.json", "reports/phase_06_jobfit_training_experiments.json"],
        "production_rule": "Keep score wording conservative until low/medium/high buckets all have validation/test coverage and bucket error is reported.",
    },
    {
        "score_name": "atsFriendliness.score",
        "range": "0-100 integer",
        "draft_bands_from_phase2": [
            {
                "band": band["band"],
                "api_min": band["api_min"],
                "api_max": band["api_max"],
                "meaning": band["product_meaning_ats"],
                "evidence_required_before_production": "Manual ATS benchmark labels, issue precision/recall, score bucket agreement, empty-text review, and calibration diagnostics.",
            }
            for band in band_policy
        ],
        "calibration_status": "draft_semantics_only_not_production_calibrated",
        "evidence_sources": ["reports/phase_02_label_schema_baselines.json", "reports/phase_07_ats_friendliness_scoring.json"],
        "production_rule": "Do not claim ATS quality band until required file cases and issue labels exist.",
    },
    {
        "score_name": "recommendations[].matchScore",
        "range": "0-100 integer",
        "draft_bands_from_phase2": [
            {
                "band": "stretch" if band["band"] == "low" else "good" if band["band"] == "medium" else "strong",
                "api_min": band["api_min"],
                "api_max": band["api_max"],
                "meaning": "Candidate-job ranking confidence for backend-supplied candidates only; label is provisional until candidate-set calibration exists.",
                "evidence_required_before_production": "Backend-like candidate sets, relevance labels, NDCG/MAP, membership constraints, and matchScore bucket error.",
            }
            for band in band_policy
        ],
        "calibration_status": "draft_semantics_only_not_production_calibrated",
        "evidence_sources": ["reports/phase_09_candidate_reranking.json", "references/docs/integrations/model-api.md"],
        "production_rule": "MatchLevel thresholds must be tied to calibrated bucket outcomes and may not be inferred from static job_index ranking.",
    },
]

calibration_diagnostics = {
    "bucket_definitions": [
        {"bucket": "0-20", "min": 0, "max": 20},
        {"bucket": "21-40", "min": 21, "max": 40},
        {"bucket": "41-60", "min": 41, "max": 60},
        {"bucket": "61-80", "min": 61, "max": 80},
        {"bucket": "81-100", "min": 81, "max": 100},
    ],
    "outputs_to_calibrate": ["jobFitAlignment.score", "atsFriendliness.score", "recommendations[].matchScore"],
    "required_tables": [
        {"name": "bucket_count", "columns": ["score_name", "split", "bucket", "row_count", "label_count"], "purpose": "Detect sparse or missing score ranges."},
        {"name": "bucket_accuracy", "columns": ["score_name", "split", "bucket", "band_agreement", "within_tolerance_rate"], "purpose": "Show how often score bucket matches label/reviewer bucket."},
        {"name": "bucket_error", "columns": ["score_name", "split", "bucket", "mae", "rmse", "mean_signed_error", "p90_absolute_error"], "purpose": "Expose over/under scoring by range."},
        {"name": "confidence_notes", "columns": ["score_name", "split", "slice", "note_key", "trigger_rule"], "purpose": "Attach score caveats to sparse labels, out-of-distribution input, parser uncertainty, or weak labels."},
    ],
    "required_plots": [
        {"plot": "reliability_curve", "x_axis": "mean_predicted_score", "y_axis": "mean_label_or_reviewer_score", "facet": "score_name"},
        {"plot": "bucket_error_bar", "x_axis": "score_bucket", "y_axis": "mae_or_signed_error", "facet": "split_and_score_name"},
        {"plot": "score_histogram", "x_axis": "predicted_score", "y_axis": "row_count", "facet": "split_and_score_name"},
        {"plot": "slice_bucket_heatmap", "x_axis": "bucket", "y_axis": "slice_value", "color": "mae_or_band_agreement"},
    ],
    "metric_definitions": [
        {"metric": "ECE", "formula": "sum(bucket_count / total_count * abs(mean_predicted_score - mean_label_score))", "use": "Global calibration error per score."},
        {"metric": "MCE", "formula": "max(abs(mean_predicted_score - mean_label_score)) across buckets", "use": "Worst bucket calibration error."},
        {"metric": "bucket_agreement", "formula": "predicted_bucket == label_or_reviewer_bucket", "use": "User-facing band stability."},
        {"metric": "within_10_points_rate", "formula": "abs(predicted_score - label_score) <= 10", "use": "Tolerance-based score reliability."},
    ],
    "current_blockers": [
        "Legacy validation/test splits have no high-fit examples under Phase 2 threshold.",
        "Production ATS calibration waits for manual benchmark labels and required file-case coverage.",
        "Recommendation matchScore calibration waits for backend-like candidate sets and relevance labels.",
        "Human-labeled validation data is not available; weak labels remain prototype-only evidence.",
    ],
}

model_card_template = {
    "schema_version": "model-card-v2-template",
    "required_fields": [
        {"field": "model.name", "type": "string", "rule": "Stable product/core model name."},
        {"field": "model.version", "type": "string", "rule": "Immutable release or experiment version."},
        {"field": "date", "type": "ISO-8601 date", "rule": "Training/export date in UTC."},
        {"field": "dataset.hash", "type": "sha256/string", "rule": "Hash or manifest hash for all training/eval data inputs."},
        {"field": "labels.version", "type": "string", "rule": "Label schema version, reviewer guideline version, and weak-label policy."},
        {"field": "split.seed", "type": "integer", "rule": "Group-safe split seed; include split artifact hash when available."},
        {"field": "features.config_version", "type": "string", "rule": "Versioned normalization, text-builder, embedding, and scoring config."},
        {"field": "metrics", "type": "object", "rule": "Global, slice, ranking, ATS, and calibration metrics with baseline references."},
        {"field": "limitations", "type": "array[string]", "rule": "Known data, label, language, parser, and deployment limitations."},
        {"field": "intended_use", "type": "array[string]", "rule": "Allowed use cases and explicit non-goals."},
    ],
    "recommended_fields": ["git_commit", "training_code_version", "artifact_manifest_path", "input_schema_version", "output_schema_version", "owner", "reviewers", "approval_status", "fallback_policy", "privacy_notes", "monitoring_notes", "calibration_report_path", "baseline_report_path", "known_blockers", "promotion_decision"],
    "metrics_required_by_output": {
        "jobFitAlignment.score": ["MAE", "RMSE", "R2", "Spearman", "score_band_agreement", "ECE", "slice_MAE"],
        "atsFriendliness.score": ["issue_precision", "issue_recall", "score_bucket_agreement", "empty_text_rate", "critical_failure_recall", "ECE"],
        "recommendations[].matchScore": ["NDCG@5", "NDCG@10", "MAP@10", "unknown_job_id_rate", "duplicate_job_id_rate", "ECE"],
    },
    "current_template_decision": "complete_template_ready_for_future_model_export; production model card cannot be marked ready until calibration evidence exists",
}


def slim_artifact(artifact: dict[str, Any]) -> dict[str, Any]:
    return {key: artifact.get(key) for key in ["path", "format", "owner", "artifact_role", "required_for_inference", "exists", "size_bytes", "sha256"] if key in artifact}

artifact_manifest_template = {
    "schema_version": "artifact-manifest-v1-template",
    "required_fields": [
        {"field": "artifact_id", "rule": "Stable unique id for each file or bundled artifact."},
        {"field": "path", "rule": "Repository or storage path."},
        {"field": "format", "rule": "File/container format such as keras, npy, parquet, json, yaml, png, ipynb."},
        {"field": "role", "rule": "inference_required, training_only, evaluation_report, config, schema, cache, or source_notebook."},
        {"field": "required_for_inference", "rule": "Boolean; true only when serving cannot run without artifact."},
        {"field": "schema_version", "rule": "Schema/config version for model input/output, labels, features, or manifest rows."},
        {"field": "sha256", "rule": "File hash; required before promotion."},
        {"field": "size_bytes", "rule": "File size for reproducibility and deploy validation."},
        {"field": "producer", "rule": "Notebook/script/run id that produced artifact."},
        {"field": "consumer", "rule": "Training, evaluation, export, or inference component that consumes artifact."},
        {"field": "reproducibility_references", "rule": "Dataset hash, label version, split seed, feature config, git commit, and model card path."},
    ],
    "current_legacy_inference_artifacts": [slim_artifact(artifact) for artifact in inference_artifacts],
    "current_training_or_audit_artifact_count": len([artifact for artifact in phase0["artifact_inventory"] if not artifact.get("required_for_inference")]),
    "hash_policy": "Every promoted artifact must have sha256 and immutable storage path; mutable cache paths are training-only unless exported with versioned manifest.",
}

deployment_readiness = {
    "current_decision": "design_complete_deployment_not_ready",
    "inference_required_artifact_classes": [
        "exported model weights or serving bundle",
        "custom layer/module code packaged outside notebooks",
        "input/output schema versions",
        "feature normalization config",
        "embedding model reference and exported candidate embedding policy if used at inference",
        "artifact manifest with hashes",
        "model card with calibration metrics and limitations",
    ],
    "training_only_artifact_classes": ["raw pair parquet", "training embeddings", "notebooks", "baseline plots", "error analysis tables", "manual review workbooks", "intermediate caches not loaded by serving"],
    "blockers_before_inference_release": sorted(set(phase5["training_readiness_gate"]["blockers"] + phase7.get("blocked_until_later_phases", []) + phase9.get("blocked_until_later_phases", []))),
    "serving_boundary_notes": [
        "Backend remains owner of auth, persistence, job detail hydration, and candidate retrieval.",
        "Model/core outputs calibrated scores and grounded signals only.",
        "Wrapper copy must not turn draft score semantics into hiring probability or production readiness claims.",
        "Invalid or missing model version, schema version, manifest hash, or calibrated score evidence blocks promotion.",
    ],
}

acceptance = {
    "score_meanings_not_hardcoded_without_report_evidence": all("evidence_sources" in item and item["calibration_status"].startswith("draft_semantics") for item in score_semantics),
    "model_card_template_is_complete": {"model.name", "model.version", "date", "dataset.hash", "labels.version", "split.seed", "features.config_version", "metrics", "limitations", "intended_use"}.issubset({field["field"] for field in model_card_template["required_fields"]}),
    "artifact_manifest_template_is_complete": {"path", "format", "role", "required_for_inference", "schema_version", "sha256", "reproducibility_references"}.issubset({field["field"] for field in artifact_manifest_template["required_fields"]}),
}

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_references": [
        "GAP_MODEL_TRAINING.md",
        "GAP_MODEL_TRAINING.md#GAP-07",
        "GAP_MODEL_TRAINING.md#GAP-09",
        "references/docs/integrations/model-api.md",
        "references/docs/generated/openapi.json",
        "reports/phase_00_reproducibility_snapshot.json",
        "reports/phase_02_label_schema_baselines.json",
        "reports/phase_05_baseline_evaluation.json",
        "reports/phase_06_jobfit_training_experiments.json",
        "reports/phase_07_ats_friendliness_scoring.json",
        "reports/phase_09_candidate_reranking.json",
    ],
    "openapi_score_constraints": score_constraints,
    "score_semantics": score_semantics,
    "calibration_diagnostics": calibration_diagnostics,
    "model_card_template": model_card_template,
    "artifact_manifest_template": artifact_manifest_template,
    "deployment_readiness": deployment_readiness,
    "blocked_until_later_phases": [
        "Production score-band wording requires calibration tables and plots with low/medium/high or 0-20 bucket coverage.",
        "Production model card cannot be marked release-ready until dataset hash, label version, split seed, feature config, metrics, limitations, and calibration report are populated from a verified run.",
        "Artifact manifest must be regenerated at export time with immutable hashes for every inference artifact.",
        "Recommendation matchScore calibration waits for backend-like candidate sets and relevance labels.",
    ],
    "acceptance": acceptance,
}

REPORTS.mkdir(exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
report["acceptance"]


{'score_meanings_not_hardcoded_without_report_evidence': True,
 'model_card_template_is_complete': True,
 'artifact_manifest_template_is_complete': True}

## Acceptance criteria

- [x] Score meanings are not hardcoded without report evidence.
- [x] Model card template is complete.
- [x] Artifact manifest template is complete.


## Verification

### Purpose
Confirm the saved report satisfies Phase 10 requirements.

### Required input
`reports/phase_10_calibration_model_card.json`.

### Action
Read the report and assert required score semantics, calibration metrics, model-card fields, artifact-manifest fields, and acceptance criteria.

### Expected output
A compact verification summary with report path, counts, and acceptance flags.

### Verification
All assertions pass.


In [9]:
saved_report = read_json(REPORT_PATH)
required_model_card_fields = {field["field"] for field in saved_report["model_card_template"]["required_fields"]}
required_manifest_fields = {field["field"] for field in saved_report["artifact_manifest_template"]["required_fields"]}
score_names = {item["score_name"] for item in saved_report["score_semantics"]}
metric_names = {metric["metric"] for metric in saved_report["calibration_diagnostics"]["metric_definitions"]}

assert saved_report["acceptance"]["score_meanings_not_hardcoded_without_report_evidence"] is True
assert saved_report["acceptance"]["model_card_template_is_complete"] is True
assert saved_report["acceptance"]["artifact_manifest_template_is_complete"] is True
assert {"jobFitAlignment.score", "atsFriendliness.score", "recommendations[].matchScore"}.issubset(score_names)
assert {"model.name", "model.version", "date", "dataset.hash", "labels.version", "split.seed", "features.config_version", "metrics", "limitations", "intended_use"}.issubset(required_model_card_fields)
assert {"path", "format", "role", "required_for_inference", "schema_version", "sha256", "reproducibility_references"}.issubset(required_manifest_fields)
assert {"ECE", "MCE", "bucket_agreement", "within_10_points_rate"}.issubset(metric_names)

verification_summary = {
    "report_path": str(REPORT_PATH.relative_to(ROOT)),
    "score_semantics_count": len(score_names),
    "calibration_bucket_count": len(saved_report["calibration_diagnostics"]["bucket_definitions"]),
    "model_card_required_field_count": len(required_model_card_fields),
    "manifest_required_field_count": len(required_manifest_fields),
    "acceptance": saved_report["acceptance"],
}
verification_summary


{'report_path': 'reports/phase_10_calibration_model_card.json',
 'score_semantics_count': 3,
 'calibration_bucket_count': 5,
 'model_card_required_field_count': 10,
 'manifest_required_field_count': 11,
 'acceptance': {'artifact_manifest_template_is_complete': True,
  'model_card_template_is_complete': True,
  'score_meanings_not_hardcoded_without_report_evidence': True}}

## Phase notes

- Production score semantics remain blocked until trusted labels/reviewer scores cover all score buckets and slices.
- Existing legacy validation/test data has no high-fit examples, so high-range score wording remains draft only.
- ATS calibration needs manual benchmark labels across required file families before release claims.
- Recommendation `matchScore` and `matchLevel` need backend-like candidate sets and relevance labels.
- Artifact manifest must be regenerated at model export time with immutable hashes for every inference artifact.
